# Lab 3 · Hyperparameter optimisation — Independent assignment

**BME · MSc Deep Learning / VITMMA19 · Fall 2026/2027**  
Department of Telecommunications and Artificial Intelligence  
**Instructor:** Dr. Mohammed Salah Al-Radhi

**60 minutes, including five minutes to check and submit · five tasks · 20 points**

### Challenge: tune a compact classifier under a fixed experiment budget

An 8×8 digit-recognition system receives noisy input. Your job is to improve a deliberately weak baseline and recommend one configuration supported by evidence. Noise is generated once per split with fixed seeds and kept identical across runs. This is **not** fresh data augmentation each epoch.

The workflow is more demanding than Lab 2: three controlled learning rates, eight automated trials, two justified refinements, and two finalists trained on three new seeds. A single lucky score does not complete the task.

**Aim for at least 85% mean validation accuracy across the three confirmation seeds**, subject to the stated budget. Still select by mean validation cross-entropy. The accuracy target is not a pass threshold and does not earn marks by itself. Full credit is available for correct methods and honest, well-supported conclusions even when the target is missed.

| Task | Minutes | Points |
| --- | ---: | ---: |
| T1 · Diagnose the baseline | 7 | 2 |
| T2 · Controlled learning-rate sweep | 10 | 4 |
| T3 · Automated search and two refinements | 18 | 6 |
| T4 · Confirm two finalists across seeds | 12 | 5 |
| T5 · Freeze, test, and interpret | 8 | 3 |
| Check and submit | 5 | — |

Submit **one executed `.ipynb` file to Moodle by the end of the lab**. Use the guided notebook and documentation as references. Follow course rules for collaboration/AI assistance and be ready to explain your work. Intentional TODOs stop the template until you complete them.

## Your details

**Name:** YOUR NAME  
**Neptun code:** YOUR NEPTUN CODE  
**Assistance used:** YOUR ANSWER

## 0. Setup and rules — provided

Save a personal copy in Drive and use a **CPU runtime**. Run these setup/helper cells first; they count within T1's time. Keep the data split, noise level, seed lists, model family, batch size, maximum epochs, and early-stopping rule unchanged.

**Selection rule:** minimise best validation cross-entropy; record accuracy as a secondary metric. Widths are 32, 64, or 128. At most 9,610 trainable parameters are allowed. AdamW is fixed. You may change only learning rate, width, dropout, and weight decay as each task permits.

**Experiment allocation:** one supplied baseline + three manual runs + eight Optuna trials + two refinements + six confirmation runs = **20 fits maximum**, each at most 25 epochs. Pruned/failed trials still consume a slot. Confirmation uses new seeds `[7, 17, 27]`; the final test uses seed **7**, chosen now, not the best-performing seed later.

Same-name reruns reuse a completed result if its configuration is unchanged. Changed settings need a permitted new slot. The log records all attempts; do not delete poor runs or restart just to erase them. A clean restart is appropriate to reproduce the **same final documented plan**.

In [ ]:
#@title Setup: install only the lab's tested Optuna version if needed
import importlib.metadata
import subprocess
import sys
try:
    optuna_version = importlib.metadata.version('optuna')
except importlib.metadata.PackageNotFoundError:
    optuna_version = None
if optuna_version != '4.7.0':
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'optuna==4.7.0'])
print("Optuna 4.7.0 is ready. Keep Colab's existing PyTorch/NumPy packages.")

In [ ]:
import copy
import hashlib
import json
import math
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch
import torch.nn as nn
from sklearn.datasets import load_digits
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from sklearn.model_selection import train_test_split
from IPython.display import display
import optuna

torch.set_num_threads(2)
DEVICE = torch.device('cpu')
SPLIT_SEED = 2026
SEARCH_SEED = 11
CONFIRM_SEEDS = [7, 17, 27]
MAX_EPOCHS = 25
BATCH_SIZE = 64
PATIENCE = 6
BASE_CONFIG = dict(lr=1e-5, hidden=64, dropout=0.0, weight_decay=1e-4)
RUNS = {}
TEST_OPENED = False
PHASE_LIMITS = dict(baseline=1, manual=3, optuna=8, refine=2, repeat=6)
optuna.logging.set_verbosity(optuna.logging.WARNING)


print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__, '| Optuna:', optuna.__version__)
print('NumPy:', np.__version__, '| scikit-learn:', sklearn.__version__)
print('Device:', DEVICE, '| threads:', torch.get_num_threads())
print('Maximum epochs:', MAX_EPOCHS, '| batch size:', BATCH_SIZE, '| patience:', PATIENCE)

In [ ]:
#@title Provided data and split helpers — expand to inspect
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def prepare_data(noise_std):
    """Fixed stratified IDs; separate fixed noise draws; no test tensors returned."""
    digits = load_digits()
    ids = np.arange(len(digits.target))
    dev_ids, test_ids = train_test_split(
        ids, test_size=0.2, stratify=digits.target, random_state=SPLIT_SEED)
    train_ids, val_ids = train_test_split(
        dev_ids, test_size=0.25, stratify=digits.target[dev_ids],
        random_state=SPLIT_SEED)
    assert set(train_ids).isdisjoint(val_ids)
    assert set(train_ids).isdisjoint(test_ids)
    assert set(val_ids).isdisjoint(test_ids)

    def tensor_split(indices, noise_seed):
        # 16 is the documented pixel maximum, not an estimate from the data.
        clean = (digits.data[indices] / 16.0).astype(np.float32)
        noise = np.random.default_rng(noise_seed).normal(0, noise_std, clean.shape)
        values = np.clip(clean + noise, 0, 1).astype(np.float32)
        return torch.from_numpy(values), torch.tensor(digits.target[indices], dtype=torch.long)

    x_train, y_train = tensor_split(train_ids, 101)
    x_val, y_val = tensor_split(val_ids, 202)
    data = dict(x_train=x_train, y_train=y_train, x_val=x_val, y_val=y_val,
                train_ids=train_ids, val_ids=val_ids, noise_std=float(noise_std),
                sizes=(len(train_ids), len(val_ids), len(test_ids)))

    def final_test(run):
        """A teaching guard, not access security. Call after freezing the choice."""
        global TEST_OPENED
        if TEST_OPENED:
            raise RuntimeError('Test already opened. Reuse final_report; do not retune.')
        if run['status'] != 'complete':
            raise ValueError('Select a completed run with its best checkpoint restored.')
        TEST_OPENED = True
        xt, yt = tensor_split(test_ids, 303)
        metrics = evaluate(run['model'], xt, yt)
        with torch.no_grad():
            predictions = run['model'](xt).argmax(1)
        return dict(loss=metrics['loss'], accuracy=metrics['acc'], n=len(yt),
                    y_true=yt.numpy(), y_pred=predictions.numpy(), images=xt.numpy(),
                    run_name=run['name'], config=copy.deepcopy(run['config']), seed=run['seed'])

    return data, final_test

def show_training_images(data):
    fig, axes = plt.subplots(2, 5, figsize=(9, 3.6))
    for i, ax in enumerate(axes.flat):
        ax.imshow(data['x_train'][i].reshape(8, 8), cmap='gray', vmin=0, vmax=1)
        ax.set_title(f"Label: {data['y_train'][i].item()}")
        ax.axis('off')
    fig.suptitle(f"Training images only | fixed noise standard deviation {data['noise_std']}")
    fig.tight_layout()
    plt.show()

In [ ]:
DATA, evaluate_final_test = prepare_data(noise_std=0.35)
print('Train / validation / held-out test:', DATA['sizes'])
print('Input shape:', tuple(DATA['x_train'].shape), '| target dtype:', DATA['y_train'].dtype)
show_training_images(DATA)

### Supplied experiment engine

The model is `64 → hidden → ReLU → dropout → 10 logits`; the maximum is **9,610 trainable parameters**. We use AdamW and cross-entropy. The engine:

1. Creates fresh model weights, optimizer state, and a seeded mini-batch order for every run.
2. Uses training data for gradients and validation loss to select an epoch.
3. Saves a **deep copy** of the best weights and restores it before returning.
4. Stops after 25 epochs, or six consecutive epochs with no lower validation loss. An Optuna pruner may stop a search trial earlier.
5. Logs every attempt, including pruned trials, to `lab3_experiments.csv`.

Training and validation curves are both measured at the end of each epoch in **evaluation mode**, with dropout off. This makes the two losses easier to compare. It adds a small evaluation cost.

The same seed gives the same initial weights when the architecture is unchanged. Different widths cannot share identical weight tensors. The split and mini-batch order still remain fixed. Seeds do not guarantee identical results across software versions or machines.

In [ ]:
#@title Provided model, evaluation, training, and experiment log
def validate_config(config):
    assert set(config) == {'lr', 'hidden', 'dropout', 'weight_decay'}
    assert 1e-5 <= config['lr'] <= 0.1, 'Keep learning rate within the allowed range.'
    assert config['hidden'] in [32, 64, 128]
    assert 0 <= config['dropout'] <= 0.5
    assert 1e-6 <= config['weight_decay'] <= 0.1

def config_key(config):
    return json.dumps(config, sort_keys=True)

def make_model(config):
    return nn.Sequential(nn.Linear(64, config['hidden']), nn.ReLU(),
                         nn.Dropout(config['dropout']), nn.Linear(config['hidden'], 10))

@torch.no_grad()
def evaluate(model, x, y):
    model.eval()
    logits = model(x)
    return {'loss': nn.functional.cross_entropy(logits, y).item(),
            'acc': (logits.argmax(1) == y).float().mean().item()}

def run_experiment(config, name, phase, seed=SEARCH_SEED, trial=None, verbose=True):
    """Fresh model/optimizer/shuffle; validation-loss selection; no test access."""
    if TEST_OPENED:
        raise RuntimeError('The test has been opened. No more tuning in this experiment.')
    validate_config(config)
    if phase not in PHASE_LIMITS:
        raise ValueError('Use one of the specified experiment phases.')
    signature = (config_key(config), int(seed), phase, MAX_EPOCHS, PATIENCE, BATCH_SIZE)
    if name in RUNS:
        run = RUNS[name]
        if run['signature'] != signature:
            raise ValueError('This name already records another experiment. Use a remaining '
                             'refinement slot, or restart to reproduce the complete plan.')
        if run['status'] == 'pruned':
            raise optuna.TrialPruned()
        if run['status'] == 'failed':
            raise FloatingPointError('This run previously failed; inspect its recorded metrics.')
        if verbose:
            print(f'{name}: reusing the recorded run; no extra training.')
        return run
    used = sum(r['phase'] == phase for r in RUNS.values())
    if used >= PHASE_LIMITS[phase]:
        raise ValueError(f'The {phase} budget ({PHASE_LIMITS[phase]} runs) is used.')
    set_seed(seed)
    model = make_model(config)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config['lr'],
                                 weight_decay=config['weight_decay'])
    generator = torch.Generator().manual_seed(seed + 1000)
    x, y = DATA['x_train'], DATA['y_train']
    history, best_state = [], None
    best_loss, best_epoch, stale, updates = float('inf'), 0, 0, 0
    status, stop_reason = 'complete', 'epoch budget'
    start = time.perf_counter()
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        indices = torch.randperm(len(y), generator=generator)
        for batch_ids in indices.split(BATCH_SIZE):
            optimizer.zero_grad(set_to_none=True)
            loss = nn.functional.cross_entropy(model(x[batch_ids]), y[batch_ids])
            if not torch.isfinite(loss):
                status, stop_reason = 'failed', 'non-finite training loss'
                break
            loss.backward()
            optimizer.step()
            updates += 1
        train = evaluate(model, x, y)
        val = evaluate(model, DATA['x_val'], DATA['y_val'])
        history.append(dict(epoch=epoch, train_loss=train['loss'], train_acc=train['acc'],
                            val_loss=val['loss'], val_acc=val['acc']))
        if status == 'failed' or not np.isfinite(list(train.values()) + list(val.values())).all():
            status, stop_reason = 'failed', 'non-finite metric'
            break
        # Keep a genuine copy. state_dict() alone contains references to live tensors.
        if val['loss'] < best_loss:
            best_loss, best_epoch, stale = val['loss'], epoch, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            stale += 1
        if trial is not None:
            trial.report(val['loss'], step=epoch)
            if trial.should_prune():
                status, stop_reason = 'pruned', 'median pruner'
                break
        if stale >= PATIENCE:
            stop_reason = f'early stopping: {PATIENCE} epochs without a new minimum'
            break
    if best_state is not None:
        model.load_state_dict(best_state)
        restored = evaluate(model, DATA['x_val'], DATA['y_val'])
        assert abs(restored['loss'] - best_loss) < 1e-6
    else:
        restored = {'loss': float('inf'), 'acc': float('nan')}
    result = dict(name=name, phase=phase, config=copy.deepcopy(config), seed=int(seed),
                  model=model, history=pd.DataFrame(history), status=status,
                  stop_reason=stop_reason, best_epoch=best_epoch,
                  best_val_loss=restored['loss'], best_val_acc=restored['acc'],
                  epochs_run=len(history), updates=updates,
                  parameters=sum(p.numel() for p in model.parameters()),
                  seconds=time.perf_counter() - start, signature=signature)
    RUNS[name] = result
    experiment_table().to_csv('lab3_experiments.csv', index=False)
    if trial is not None:
        for key in ['name', 'best_epoch', 'epochs_run', 'updates', 'best_val_acc', 'stop_reason']:
            trial.set_user_attr(key, result[key])
    if verbose:
        print(f"{name}: {status}; best validation CE {result['best_val_loss']:.4f}; "
              f"accuracy {result['best_val_acc']:.3f}; epoch {best_epoch}/{len(history)}; "
              f"{result['seconds']:.2f} s")
    if status == 'pruned':
        raise optuna.TrialPruned()
    if status == 'failed':
        raise FloatingPointError(stop_reason)
    return result

def experiment_table():
    rows = []
    for r in RUNS.values():
        keys = ['name', 'phase', 'seed', 'status', 'best_val_loss', 'best_val_acc',
                'best_epoch', 'epochs_run', 'updates', 'parameters', 'seconds', 'stop_reason']
        rows.append({**{k: r[k] for k in keys}, **r['config']})
    return pd.DataFrame(rows)

In [ ]:
#@title Provided plotting helpers
def plot_curves(runs, title):
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    for run in runs:
        h = run['history']
        line, = axes[0].plot(h.epoch, h.val_loss, label=run['name'] + ' validation')
        axes[0].plot(h.epoch, h.train_loss, '--', color=line.get_color(), alpha=.65)
        axes[1].plot(h.epoch, h.val_acc, label=run['name'])
        axes[0].scatter(run['best_epoch'], run['best_val_loss'], color=line.get_color(), s=25)
    axes[0].set(xlabel='Epoch', ylabel='Cross-entropy', title='Solid: validation | dashed: training')
    axes[1].set(xlabel='Epoch', ylabel='Validation accuracy', ylim=(0, 1.02))
    for ax in axes:
        ax.grid(alpha=.2)
        ax.legend(fontsize=8)
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()

def plot_search(study):
    trials = study.trials_dataframe()
    complete = trials[trials.state == 'COMPLETE'].copy()
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.6))
    if len(complete):
        axes[0].scatter(complete.number, complete.value, label='Completed trial')
        axes[0].plot(complete.number, complete.value.cummin(), label='Best so far')
        axes[1].scatter(complete.params_lr, complete.value)
        axes[1].set_xscale('log')
    axes[0].set(xlabel='Trial number', ylabel='Best validation CE')
    axes[1].set(xlabel='Learning rate (log scale)', ylabel='Best validation CE')
    axes[0].legend()
    for ax in axes: ax.grid(alpha=.2)
    fig.tight_layout()
    plt.show()
    print('Only COMPLETE trials are ranked here. PRUNED trials remain in the experiment log.')

def show_test_report(report):
    print(f"Selected run: {report['run_name']} | test n={report['n']} | "
          f"CE={report['loss']:.4f} | accuracy={report['accuracy']:.3f}")
    cm = confusion_matrix(report['y_true'], report['y_pred'], labels=np.arange(10))
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay(cm, display_labels=np.arange(10)).plot(ax=ax, colorbar=False)
    ax.set_title('Held-out test digits | rows=true, columns=predicted')
    fig.tight_layout()
    plt.show()
    errors = np.flatnonzero(report['y_true'] != report['y_pred'])[:6]
    if len(errors):
        fig, axes = plt.subplots(1, len(errors), figsize=(2 * len(errors), 2.3), squeeze=False)
        for ax, idx in zip(axes.flat, errors):
            ax.imshow(report['images'][idx].reshape(8, 8), cmap='gray', vmin=0, vmax=1)
            ax.set_title(f"True {report['y_true'][idx]} / pred {report['y_pred'][idx]}", fontsize=9)
            ax.axis('off')
        fig.tight_layout()
        plt.show()
    else:
        print('No mistakes in this test split.')

## T1 · Diagnose before changing anything · 2 points · 7 minutes

Run the baseline below. It has a deliberately small learning rate. Record the first/last training loss and best validation result. Use both curves to explain whether the immediate problem is weak optimisation, overfitting, or insufficient evidence to distinguish causes.

Also explain why the test set cannot guide any of T1–T4. **You do not need to improve the model in T1.**

In [ ]:
baseline = run_experiment(BASE_CONFIG, 'Baseline', 'baseline')
display(baseline['history'].iloc[[0, -1]])
plot_curves([baseline], 'Baseline diagnosis')

### T1 · Your diagnosis

**Observed losses and validation score:** YOUR VALUES  
**Diagnosis and supporting evidence:** YOUR ANSWER (2–3 sentences)  
**Why the test must stay closed:** YOUR ANSWER

## T2 · Controlled learning-rate experiment · 4 points · 10 minutes

Choose **three distinct learning rates** in `[1e-4, 1e-1]`, spanning at least a factor of 10. Keep every other baseline setting unchanged. State a prediction before running them, then compare the curves and results.

Explain why orders of magnitude are a useful first search. Choose the best setting using validation **loss**, and report its accuracy as well. State a limitation of a one-seed comparison.

**Deliverables:** completed code, three-run table, curves, prediction, and a short evidence-based conclusion.

### T2 · Prediction before execution

**My three rates:** YOUR VALUES  
**Why these values, and which I expect to work best:** YOUR ANSWER

In [ ]:
MANUAL_LRS = [None, None, None]  # TODO: three chosen rates
assert all(isinstance(v, (float, int)) for v in MANUAL_LRS), 'Complete MANUAL_LRS first.'
assert len(set(MANUAL_LRS)) == 3 and all(1e-4 <= v <= 1e-1 for v in MANUAL_LRS)
assert max(MANUAL_LRS) / min(MANUAL_LRS) >= 10
manual_runs = [run_experiment({**BASE_CONFIG, 'lr': lr}, f'Manual_{i+1}', 'manual')
               for i, lr in enumerate(MANUAL_LRS)]
display(experiment_table().query("phase == 'manual'")[
    ['name', 'lr', 'best_val_loss', 'best_val_acc', 'best_epoch', 'epochs_run']])
plot_curves(manual_runs, 'T2: learning-rate sweep')

### T2 · Results and interpretation

**Chosen run / validation CE / validation accuracy:** YOUR VALUES  
**What the curves show, and whether my prediction held:** YOUR ANSWER  
**Why this is a controlled comparison, and one remaining limitation:** YOUR ANSWER

## T3 · Search, inspect, then refine · 6 points · 18 minutes

**A. Define a search space from T2's evidence.** Choose positive LR bounds inside `[1e-5, 1e-1]`, spanning at least a factor of 10. Explain whether you broaden or narrow the search and why. Complete `objective` to tune four hyperparameters:

| Hyperparameter | Allowed search |
| --- | --- |
| Learning rate | Your bounds; logarithmic |
| Hidden width | Categorical: 32, 64, 128 |
| Dropout | Uniform from 0.0 to 0.5 |
| Weight decay | Logarithmic from 1e-6 to 1e-1 |

Use exactly eight TPE trials with four startup trials. Keep the supplied pruner. Return the restored run's **best validation loss**. Choose the correct optimisation direction.

**B. Inspect the log.** Report completed/pruned/failed counts and actual epochs versus the 200-epoch upper bound. Explain why a pruned partial score is not ranked as a finished candidate and why unused epochs are not a pruning-only speedup.

**C. Make two informed refinements.** After viewing the search, the provided cell selects the best completed manual/Optuna configuration as a parent. Before running two new experiments, propose **one changed hyperparameter per experiment**, explain the observation motivating each, and predict the outcome. An unsuccessful refinement is valid evidence.

**Deliverables:** working objective, search table/plot, budget account, two pre-run hypotheses, two completed follow-ups, and an interpretation. Do not just copy a winning configuration without reasoning.

### T3A · Search plan before execution

**LR lower/upper bounds:** YOUR VALUES  
**Evidence from T2 supporting these bounds:** YOUR ANSWER  
**Why log sampling for LR/weight decay but linear sampling for dropout:** YOUR ANSWER

In [ ]:
LR_LOW, LR_HIGH = None, None  # TODO
assert isinstance(LR_LOW, (float, int)) and isinstance(LR_HIGH, (float, int))
assert 1e-5 <= LR_LOW < LR_HIGH <= 1e-1 and LR_HIGH / LR_LOW >= 10

def objective(trial):
    # TODO: use trial.suggest_float and trial.suggest_categorical.
    # See the guided example, but justify this assignment's learning-rate bounds.
    config = dict(lr=None, hidden=None, dropout=None, weight_decay=None)
    if any(value is None for value in config.values()):
        raise NotImplementedError('Complete the four hyperparameter suggestions.')
    run = run_experiment(config, f'TPE_{trial.number:02d}', 'optuna', trial=trial)
    return None  # TODO: return the validation metric being optimised

DIRECTION = None  # TODO: 'minimize' or 'maximize'
assert DIRECTION in ['minimize', 'maximize']
if 'study' not in globals():
    study = optuna.create_study(
        direction=DIRECTION,
        sampler=optuna.samplers.TPESampler(seed=42, n_startup_trials=4),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=5))
study.optimize(objective, n_trials=max(0, 8-len(study.trials)), n_jobs=1,
               catch=(FloatingPointError,))
display(study.trials_dataframe()[['number', 'state', 'value', 'params_lr',
                                 'params_hidden', 'params_dropout', 'params_weight_decay']])
plot_search(study)

In [ ]:
search_log = experiment_table().query("phase == 'optuna'")
display(search_log[['name', 'status', 'lr', 'hidden', 'dropout', 'weight_decay',
                    'best_val_loss', 'best_val_acc', 'epochs_run', 'seconds']])
print('States:', search_log.status.value_counts().to_dict())
print('Actual epochs:', int(search_log.epochs_run.sum()), '| upper allocation:', 8 * MAX_EPOCHS)
print('Actual recorded seconds:', round(search_log.seconds.sum(), 2))
candidate_table = experiment_table().query("phase in ['manual', 'optuna'] and status == 'complete'")
parent_name = candidate_table.sort_values(['best_val_loss', 'parameters', 'name']).iloc[0]['name']
parent = RUNS[parent_name]
print('Parent for both refinements:', parent_name, parent['config'])

### T3B/C · Evidence and hypotheses before refinement

**Completed / pruned / failed trials; actual / maximum epochs:** YOUR VALUES  
**Why partial trials and unused epochs need careful interpretation:** YOUR ANSWER  
**Refinement 1 — one change, observation motivating it, predicted effect:** YOUR ANSWER  
**Refinement 2 — one change, observation motivating it, predicted effect:** YOUR ANSWER

In [ ]:
# TODO: each entry is a one-key dictionary, e.g. {'dropout': your_value}.
# Both are changes from the displayed parent, not sequential changes from each other.
REFINE_CHANGES = [None, None]
assert len(REFINE_CHANGES) == 2
refinement_runs = []
for i, change in enumerate(REFINE_CHANGES):
    assert isinstance(change, dict) and len(change) == 1, 'Choose exactly one change.'
    key, value = next(iter(change.items()))
    assert key in parent['config'] and value != parent['config'][key]
    cfg = {**parent['config'], **change}
    refinement_runs.append(run_experiment(cfg, f'Refine_{i+1}', 'refine'))
assert config_key(refinement_runs[0]['config']) != config_key(refinement_runs[1]['config'])
display(experiment_table().query("phase == 'refine'"))
plot_curves([parent] + refinement_runs, 'T3: parent and two one-variable refinements')

### T3 · What changed?

**Parent and refinement CE/accuracy:** YOUR VALUES  
**Did either refinement help? What does this suggest for a future experiment?** YOUR ANSWER (3–4 sentences)

Do not start an extra experiment after the allocation is used. Propose it in writing.

## T4 · Is the winner robust to training randomness? · 5 points · 12 minutes

Choose **two distinct completed configurations** from T2/T3, using validation evidence only. The first should be your best candidate; the second should be a credible alternative (for example, close validation loss or a smaller model). Justify both choices.

Train each from scratch with the same three new seeds, `[7, 17, 27]`, using the supplied loop. Build a two-row summary containing mean validation CE, **sample standard deviation** (`ddof=1`), mean accuracy, its sample standard deviation, and parameter count.

Select the configuration with lower **mean validation CE**; exact ties use fewer parameters, then candidate name. Do not change this rule after seeing test data. Discuss one case where seed variation changes or weakens the single-run conclusion.

**Important:** three seeds vary initialisation/dropout/shuffling on the same data split. They are not cross-validation, a confidence interval, or proof of statistical significance.

In [ ]:
available = experiment_table().query("phase in ['manual', 'optuna', 'refine'] and status == 'complete'")
display(available.sort_values('best_val_loss')[
    ['name', 'lr', 'hidden', 'dropout', 'weight_decay', 'best_val_loss', 'best_val_acc', 'parameters']])

### T4 · Finalist choices before confirmation

**Candidate A run name / reason:** YOUR ANSWER  
**Candidate B run name / reason:** YOUR ANSWER

In [ ]:
FINALIST_NAMES = [None, None]  # TODO: names from the table, with distinct configurations
assert len(FINALIST_NAMES) == 2 and all(n in available.name.values for n in FINALIST_NAMES)
finalist_configs = {label: copy.deepcopy(RUNS[name]['config'])
                    for label, name in zip(['A', 'B'], FINALIST_NAMES)}
assert config_key(finalist_configs['A']) != config_key(finalist_configs['B'])
confirmation_rows = []
for label, config in finalist_configs.items():
    for seed in CONFIRM_SEEDS:
        r = run_experiment(config, f'Confirm_{label}_s{seed}', 'repeat', seed=seed)
        confirmation_rows.append(dict(candidate=label, seed=seed, val_loss=r['best_val_loss'],
                                      val_acc=r['best_val_acc'], parameters=r['parameters']))
confirmation = pd.DataFrame(confirmation_rows)
display(confirmation)

In [ ]:
# TODO: aggregate by candidate. Pandas groupby.std() uses ddof=1 by default.
# Required columns: candidate, mean_val_loss, std_val_loss,
#                   mean_val_acc, std_val_acc, parameters
summary = None
assert isinstance(summary, pd.DataFrame), 'Complete the two-row summary.'
required = {'candidate', 'mean_val_loss', 'std_val_loss', 'mean_val_acc', 'std_val_acc', 'parameters'}
assert required.issubset(summary.columns) and len(summary) == 2
display(summary)
winner = summary.sort_values(['mean_val_loss', 'parameters', 'candidate']).iloc[0]['candidate']
selected_run = RUNS[f'Confirm_{winner}_s7']  # The test seed was fixed before any tuning.
print('Chosen configuration:', winner, selected_run['config'])
print('Predeclared final seed:', selected_run['seed'], '| restored epoch:', selected_run['best_epoch'])

### T4 · Your conclusion

**Winner and the two mean ± sample-SD validation losses:** YOUR VALUES  
**Does seed variation support a clear winner? Use the six results:** YOUR ANSWER  
**Why these three runs do not measure uncertainty over new datasets:** YOUR ANSWER

## T5 · Freeze the decision, then test once · 3 points · 8 minutes

Before opening the test set, write a short decision explaining the chosen configuration, the validation evidence, and one limitation. The final model is the restored **seed-7 checkpoint** of the winning configuration. No further training or configuration changes are allowed after this point.

Run the provided evaluation and discuss one confusion/error using the images. Report the test sample count and accuracy and compare cautiously with validation. **Do not use this result to choose another configuration.** This is an artificial noisy-digits split, not a claim about deployed handwriting recognition.

### T5 · Decision before test access

**Selected configuration, evidence, and limitation:** YOUR ANSWER (3–4 sentences)

In [ ]:
DECISION_WRITTEN = False  # TODO: set True only after writing the decision above.
assert DECISION_WRITTEN, 'Write and freeze the decision before opening the test.'
if 'final_report' not in globals():
    final_report = evaluate_final_test(selected_run)
show_test_report(final_report)

### T5 · Final interpretation

**Test count / accuracy / CE:** YOUR VALUES  
**One observed confusion and a plausible explanation using the images:** YOUR ANSWER  
**What the validation/test difference can and cannot tell us:** YOUR ANSWER  
**If more time were available, one next experiment (proposal only):** YOUR ANSWER

In [ ]:
# Required final evidence: keep this table in the submitted notebook.
final_log = experiment_table()
display(final_log[['name', 'phase', 'seed', 'status', 'best_val_loss', 'best_val_acc',
                   'best_epoch', 'epochs_run', 'parameters']].round(5))
display(final_log[['name', 'lr', 'hidden', 'dropout', 'weight_decay']].round(7))
print('Total recorded fits:', len(final_log), '| maximum allocation: 20')
print('Total epochs:', int(final_log.epochs_run.sum()), '| maximum allocation:', 20 * MAX_EPOCHS)
print('CSV saved locally as lab3_experiments.csv; the visible table is sufficient for submission.')
assert len(final_log) <= 20
assert set(confirmation.seed) == set(CONFIRM_SEEDS)
assert selected_run['seed'] == 7

## Check and submit · final 5 minutes

1. Fill in your name, Neptun code, and assistance used. Replace every `YOUR ANSWER`/`YOUR VALUES` prompt and complete the TODOs.
2. Keep all attempted runs, written predictions, result tables, learning curves, search plot, six-seed results, summary, confusion matrix, and final decision visible. Failed refinements remain useful evidence.
3. Check the final notebook from top to bottom. If you restart and rerun, reproduce the **same frozen plan**; do not retune using the already-seen test score. The untouched template intentionally stops at TODOs.
4. Rename it **`YOUR_NEPTUN_DL_Lab03.ipynb`**, then choose **File → Download → Download .ipynb**. Keep outputs included when saving.
5. Upload the executed notebook to **Moodle before the lab ends** and confirm submission. A Colab link alone is insufficient. The CSV is an optional backup, not a second required file.

If incomplete, submit your partial work on time and explain the blocker honestly. If Moodle access is unavailable, keep the downloaded file, inform the instructor before the deadline, and email the work as specified in the course README.

Keep completed answers and personal details out of the public GitHub repository.

**Optional extension after submission:** compare random search with TPE under matched budgets on a fresh development exercise, or explore a learning-rate schedule. Do not reopen this submission's test set for tuning.

## References and credits

- [scikit-learn: the 8×8 digits dataset](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html), derived from UCI Optical Recognition of Handwritten Digits, C. Kaynak. This is **not MNIST**.
- [PyTorch: AdamW](https://docs.pytorch.org/docs/stable/generated/torch.optim.AdamW.html), [saving best-model copies](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html), and [reproducibility](https://docs.pytorch.org/docs/stable/notes/randomness.html).
- [Optuna: search and pruning](https://optuna.readthedocs.io/en/v4.7.0/tutorial/10_key_features/003_efficient_optimization_algorithms.html), [TPE sampler](https://optuna.readthedocs.io/en/v4.7.0/reference/samplers/generated/optuna.samplers.TPESampler.html), and [median pruner](https://optuna.readthedocs.io/en/v4.7.0/reference/generated/optuna.pruners.MedianPruner.html).
- Bergstra & Bengio, [Random Search for Hyper-Parameter Optimization](https://www.jmlr.org/papers/v13/bergstra12a.html), JMLR, 2012.
- BME lecture: *Deep Learning — Hyperparameter Optimization* (29 September 2026), supplied course slides.

Course material: Dr. Mohammed Salah Al-Radhi, BME. Dataset and framework credits remain with their authors. The added noise is an artificial classroom transformation, not a new benchmark dataset.